In [3]:
import json
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [4]:
# Base project directory (current working directory)
print("Current working directory:", os.getcwd())

# Path to VIA training annotation file
VIA_JSON_PATH = "../data/0Train_via_annos.json"

# (Optional sanity check)
print("VIA file exists:", os.path.exists(VIA_JSON_PATH))


Current working directory: c:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\notebooks
VIA file exists: True


In [5]:
# Paths
TRAIN_IMG_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\train_images"
VIA_JSON_PATH = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\0Train_via_annos.json"


In [6]:
with open(VIA_JSON_PATH, "r") as f:
    via_data = json.load(f)

print("Loaded VIA annotations for", len(via_data), "images")


Loaded VIA annotations for 11621 images


In [7]:
# Inspect one sample entry to understand VIA structure
first_key = list(via_data.keys())[0]
print("Sample key:", first_key)
print("Sample value:")
print(via_data[first_key])


Sample key: 01012020_172204image853193.jpg
Sample value:
{'name': '01012020_172204image853193.jpg', 'regions': [{'all_x': [1, 30, 81, 79, 74, 65, 63, 65, 83, 100, 126, 187, 208, 228, 300, 351, 394, 447, 438, 425, 408, 389, 373, 301, 258, 211, 189, 189, 2], 'all_y': [107, 118, 184, 157, 131, 98, 86, 83, 110, 165, 245, 417, 464, 481, 535, 554, 568, 578, 444, 392, 336, 256, 191, 171, 135, 69, 22, 1, 1], 'class': 'mat_bo_phan'}, {'all_x': [395, 457, 528, 621, 667, 705, 752, 772, 792, 798, 799, 775, 754, 728, 710, 659, 579, 506, 485, 479, 389], 'all_y': [256, 264, 267, 267, 268, 277, 294, 301, 300, 296, 247, 264, 272, 276, 270, 243, 251, 255, 250, 245, 231], 'class': 'rach'}, {'all_x': [440, 461, 485, 509, 548, 620, 695, 737, 738, 727, 673, 625, 596, 555, 449], 'all_y': [443, 450, 443, 443, 440, 426, 411, 405, 408, 424, 456, 479, 493, 504, 512], 'class': 'rach'}, {'all_x': [413, 438, 438, 429], 'all_y': [340, 364, 387, 400], 'class': 'mop_lom'}, {'all_x': [464, 446, 441, 438, 432, 444, 470,

In [8]:
# Map class names to numeric labels
CLASS_MAP = {
    "mat_bo_phan": 1,
    "rach": 2,
    "mop_lom": 3,
    "tray_son": 4
}

print("Class mapping:", CLASS_MAP)


Class mapping: {'mat_bo_phan': 1, 'rach': 2, 'mop_lom': 3, 'tray_son': 4}


In [9]:
# Pick one sample
sample_key = list(via_data.keys())[0]
sample = via_data[sample_key]

image_path = os.path.join(TRAIN_IMG_DIR, sample["name"])
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("Image shape:", image.shape)
print("Number of damage regions:", len(sample["regions"]))


Image shape: (600, 800, 3)
Number of damage regions: 6


In [10]:
height, width, _ = image.shape

# One mask per region
masks = []

for region in sample["regions"]:
    xs = region["all_x"]
    ys = region["all_y"]
    
    polygon = np.array(list(zip(xs, ys)), dtype=np.int32)
    
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [polygon], 1)
    
    masks.append(mask)

masks = np.array(masks)
print("Masks shape:", masks.shape)


Masks shape: (6, 600, 800)


In [11]:
train_annotations = []

for _, value in via_data.items():
    img_name = value["name"]
    regions = value["regions"]

    boxes = []
    labels = []

    for region in regions:
        all_x = region["all_x"]
        all_y = region["all_y"]
        cls_name = region["class"]

        # Skip unknown classes (safety)
        if cls_name not in CLASS_MAP:
            continue

        # Polygon → bounding box
        x_min = float(min(all_x))
        y_min = float(min(all_y))
        x_max = float(max(all_x))
        y_max = float(max(all_y))

        # Validate box
        if x_max > x_min and y_max > y_min:
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(CLASS_MAP[cls_name])

    # Only keep images with at least one valid box
    if len(boxes) > 0:
        train_annotations.append({
            "image_name": img_name,
            "boxes": boxes,
            "labels": labels
        })

print("Usable training samples:", len(train_annotations))


Usable training samples: 9580


In [12]:
# Verify annotations are JSON-serializable
json.dumps(train_annotations)
print("train_annotations is JSON-safe ✔")


train_annotations is JSON-safe ✔


In [13]:
# Save parsed annotations
OUTPUT_JSON = "../data/train_annotations.json"

with open(OUTPUT_JSON, "w") as f:
    json.dump(train_annotations, f, indent=2)

print("Saved:", OUTPUT_JSON)


Saved: ../data/train_annotations.json
